In [ ]:
import os, json, pandas as pd, numpy as np, joblib, matplotlib.pyplot as plt
import thermoift.PLOT_SETTINGS as ps
from thermoift import MLPostprocessing, plot_correlation_heatmap
from matplotlib.ticker import AutoMinorLocator

In [ ]:
PLOT_FOLDER = "RF_TabPFN_P_bubble_OUTPUTS"
target      = "P_bubble"

In [ ]:
with open(os.path.join(PLOT_FOLDER, f"RF_TabPFN_{target}_metrics.json")) as f:
    metrics = json.load(f)

features     = metrics["features"]
cv_r2_mean   = metrics["cv_r2_mean"]
cv_rmse_mean = metrics["cv_rmse_mean"]
cv_mae_mean  = metrics["cv_mae_mean"]

preds_df = pd.read_csv(os.path.join(PLOT_FOLDER, f"RF_TabPFN_{target}_predictions.csv"))

print(f"Loaded artifacts from {PLOT_FOLDER}/")
print(f"Features : {features}")
print(f"CV R²    : {cv_r2_mean:.6f}  |  CV RMSE: {cv_rmse_mean:.6f}  |  CV MAE: {cv_mae_mean:.6f}")

In [ ]:
train_rows = preds_df[preds_df["split"] == "train"]
test_rows  = preds_df[preds_df["split"] == "test"]
val_rows   = preds_df[preds_df["split"] == "val"]

y_train      = pd.Series(train_rows["actual"].values, name=target)
y_train_pred = train_rows["predicted"].values
y_test       = pd.Series(test_rows["actual"].values, index=test_rows["idx"].values, name=target)
y_test_pred  = test_rows["predicted"].values
y_val        = pd.Series(val_rows["actual"].values, name=target)
y_val_pred   = val_rows["predicted"].values

post = MLPostprocessing(
    y_true=y_test,
    y_pred=y_test_pred,
    target=target,
    feature_names=features,
    datasets={
        "train": (y_train, y_train_pred),
        "test":  (y_test,  y_test_pred),
        "val":   (y_val,   y_val_pred),
    },
)

In [ ]:
df = pd.read_csv("../interfacial_results_dataset_A4.csv")
plot_correlation_heatmap(df, features, target, save_path=f"RF_TabPFN_{target}_correlation", folder=PLOT_FOLDER)

In [ ]:
post.plot_parity(model_name="RF_TabPFN", save_path=f"RF_TabPFN_{target}_parity_plot", folder=PLOT_FOLDER,
                 cv_r2=cv_r2_mean, cv_rmse=cv_rmse_mean, cv_mae=cv_mae_mean)

In [ ]:
post.plot_residual_distribution(save_path=f"RF_TabPFN_{target}_residual_distribution", folder=PLOT_FOLDER,
                                cv_r2=cv_r2_mean, cv_rmse=cv_rmse_mean, cv_mae=cv_mae_mean)

In [ ]:
post.plot_residual_vs_predicted(save_path=f"RF_TabPFN_{target}_residual_vs_predicted", folder=PLOT_FOLDER,
                                cv_r2=cv_r2_mean, cv_rmse=cv_rmse_mean, cv_mae=cv_mae_mean)

In [ ]:
nestimator_conv_path = os.path.join(PLOT_FOLDER, f"RF_TabPFN_{target}_nestimators_convergence.csv")

if os.path.exists(nestimator_conv_path):
    nestimator_conv_df = pd.read_csv(nestimator_conv_path)

    fig, ax = ps.plot_init(w=5, h=3.5)
    ax.plot(
        nestimator_conv_df["n_estimators"],
        nestimator_conv_df["train_rmse"],
        marker="o",
        markersize=ps.markersize,
        lw=ps.linewidth,
        color=ps.colors[2],
        label="Train",
    )
    ax.plot(
        nestimator_conv_df["n_estimators"],
        nestimator_conv_df["cv_rmse"],
        marker="^",
        markersize=ps.markersize,
        lw=ps.linewidth,
        color=ps.colors[0],
        label="CV",
    )

    selected_n_estimators = metrics.get("n_estimators", 10)
    ax.axvline(
        selected_n_estimators,
        ls="--",
        color="gray",
        lw=ps.linewidth,
        label="Selected (n_estimators)",
    )

    ax.set_xlabel(r"$n_{\mathrm{estimators}}$", fontsize=ps.label_fontsize)
    ax.set_ylabel(r"RMSE", fontsize=ps.label_fontsize)
    ax.minorticks_on()
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.tick_params(axis="both", which="minor", length=3)
    ps.style_legend(ax, loc="upper right")
    plt.tight_layout()
    ps.save_plot(fig, f"RF_TabPFN_{target}_nestimators_convergence", folder=PLOT_FOLDER)
    plt.show()
    print(f"Saved: RF_TabPFN_{target}_nestimators_convergence")
else:
    print(f"Missing n_estimators convergence file: {nestimator_conv_path}")


In [ ]:
max_depth_conv_path = os.path.join(PLOT_FOLDER, f"RF_TabPFN_{target}_max_depth_convergence.csv")

if os.path.exists(max_depth_conv_path):
    max_depth_conv_df = pd.read_csv(max_depth_conv_path)
    depth_labels = max_depth_conv_df["max_depth"].astype(str).tolist()
    x = np.arange(len(depth_labels))

    fig, ax = ps.plot_init(w=5, h=3.5)
    ax.plot(
        x,
        max_depth_conv_df["train_rmse"],
        marker="o",
        markersize=ps.markersize,
        lw=ps.linewidth,
        color=ps.colors[2],
        label="Train",
    )
    ax.plot(
        x,
        max_depth_conv_df["cv_rmse"],
        marker="^",
        markersize=ps.markersize,
        lw=ps.linewidth,
        color=ps.colors[0],
        label="CV",
    )

    selected_depth = str(metrics.get("max_depth", 3))
    if selected_depth in depth_labels:
        ax.axvline(
            depth_labels.index(selected_depth),
            ls="--",
            color="gray",
            lw=ps.linewidth,
            label="Selected (max_depth)",
        )

    ax.set_xticks(x)
    ax.set_xticklabels(depth_labels)
    ax.set_xlabel(r"$\mathrm{max\ depth}$", fontsize=ps.label_fontsize)
    ax.set_ylabel(r"RMSE", fontsize=ps.label_fontsize)
    ax.minorticks_on()
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.tick_params(axis="both", which="minor", length=3)
    ps.style_legend(ax, loc="upper right")
    plt.tight_layout()
    ps.save_plot(fig, f"RF_TabPFN_{target}_max_depth_convergence", folder=PLOT_FOLDER)
    plt.show()
    print(f"Saved: RF_TabPFN_{target}_max_depth_convergence")
else:
    print(f"Missing max_depth convergence file: {max_depth_conv_path}")


In [ ]:
post.print_summary()